In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb


# Load output csv from previous collab notebook ('merged_df_with_summary_best_classes.csv')
df = pd.read_csv('newest_df.csv')
df.drop(df.columns[[0]], axis=1, inplace=True)        # drop id

# split
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
target = 'Rating_y_cleaned'
numeric_cols.remove(target)

X = df[numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)


# def report funcs

def regression_report(y_true, y_pred):
    mse  = mean_squared_error(y_true, y_pred)
    return {
        "MAE" : mean_absolute_error(y_true, y_pred),
        "MSE" : mse,
        "RMSE": np.sqrt(mse),
        "R2"  : r2_score(y_true, y_pred)
    }

def cv_report(model, X, y, cv):
    scoring = {
        "MAE": "neg_mean_absolute_error",
        "MSE": "neg_mean_squared_error",
        "R2" : "r2"
    }
    cv_res = cross_validate(model, X, y, cv=cv, scoring=scoring,
                            n_jobs=-1, return_train_score=False)
    mae  = -cv_res["test_MAE"].mean()
    mse  = -cv_res["test_MSE"].mean()
    rmse = np.sqrt(mse)
    r2   =  cv_res["test_R2"].mean()
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
top_k     = 15
rnd_state = 42


# RF Regressor

rf_model = RandomForestRegressor(random_state=rnd_state)
rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_test_pred  = rf_model.predict(X_test)

rf_metrics_train = regression_report(y_train, rf_train_pred)
rf_metrics_test  = regression_report(y_test,  rf_test_pred)
rf_metrics_cv    = cv_report(rf_model, X, y, cv)

rf_imp = (
    pd.Series(rf_model.feature_importances_, index=numeric_cols)
      .sort_values(ascending=False)
      .head(top_k)
)


# XGBoost regressor

xgb_model = xgb.XGBRegressor(
    random_state=rnd_state,
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6
)
xgb_model.fit(X_train, y_train)

xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred  = xgb_model.predict(X_test)

xgb_metrics_train = regression_report(y_train, xgb_train_pred)
xgb_metrics_test  = regression_report(y_test,  xgb_test_pred)
xgb_metrics_cv    = cv_report(xgb_model, X, y, cv)

xgb_imp = (
    pd.Series(xgb_model.feature_importances_, index=numeric_cols)
      .sort_values(ascending=False)
      .head(top_k)
)


# results

print("\n=== Random Forest Metrics ===")
print("Train :", rf_metrics_train)
print("Test  :", rf_metrics_test)
print("CV(5) :", rf_metrics_cv)

print("\nTop‑{} RF feature importances:".format(top_k))
print(rf_imp.to_string(float_format="%.4f"))

print("\n=== XGBoost Metrics ===")
print("Train :", xgb_metrics_train)
print("Test  :", xgb_metrics_test)
print("CV(5) :", xgb_metrics_cv)

print("\nTop‑{} XGB feature importances:".format(top_k))
print(xgb_imp.to_string(float_format="%.4f"))



=== Random Forest Metrics ===
Train : {'MAE': 0.03047392204411052, 'MSE': 0.00766789272814792, 'RMSE': np.float64(0.08756650460163361), 'R2': 0.9941754208858475}
Test  : {'MAE': 0.08153282017152454, 'MSE': 0.051376635531184836, 'RMSE': np.float64(0.22666414699105997), 'R2': 0.959146265084658}
CV(5) : {'MAE': np.float64(0.07570416165755305), 'MSE': np.float64(0.04672035672570155), 'RMSE': np.float64(0.21614892256428564), 'R2': np.float64(0.9639346099463936)}

Top‑15 RF feature importances:
bert_binary_acc_summary   0.2533
bert_binary_f1_summary    0.2530
bert_binary_acc_review    0.1802
User rating_scaled        0.0521
vader_score_summary       0.0469
Like count_scaled         0.0438
vader_score               0.0410
Comment count_scaled      0.0376
Website rating            0.0351
review_length             0.0208
review_word_count         0.0134
bert_sentiment_numeric    0.0039
budget_scaled             0.0038
popularity_scaled         0.0030
runtime_scaled            0.0030

=== XGBoo